# BERT and Pretraining

## Learning Objectives
1. Understand how Masked Language Modeling (MLM) masking strategy works at the token level
2. Implement a minimal BERT-like transformer with MLM pretraining in PyTorch
3. Analyze the effect of pretraining on downstream task performance vs random initialization
4. Compare sentence embedding strategies (CLS token vs mean-pooling) for feature extraction

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Level 1: MLM Masking Strategy in NumPy

Masked Language Modeling (MLM) is the core pretraining objective of BERT. Given a sequence of
token IDs, 15% are selected as candidates. Of those candidates:
- 80% are replaced with the special `[MASK]` token (id=0)
- 10% are replaced with a random token from the vocabulary
- 10% are kept unchanged

This mixed strategy prevents the model from always seeing `[MASK]` at inference time, since
real text never contains mask tokens.

In [ ]:
# Level 1: MLM Masking in NumPy
MASK_TOKEN_ID = 0
VOCAB_SIZE = 100
MASK_RATIO = 0.15

def apply_mlm_masking(token_ids: list, vocab_size: int = VOCAB_SIZE,
                       mask_token_id: int = MASK_TOKEN_ID,
                       mask_ratio: float = MASK_RATIO,
                       seed: int = None) -> dict:
    """
    Apply BERT-style MLM masking to a token sequence.

    Args:
        token_ids: List of integer token ids (1..vocab_size-1; 0 reserved for MASK)
        vocab_size: Size of vocabulary
        mask_token_id: ID used for the [MASK] token
        mask_ratio: Fraction of tokens to select as candidates (default 0.15)
        seed: Optional random seed for reproducibility

    Returns:
        dict with keys:
          'input_ids'       - masked sequence fed to the model
          'labels'          - original token ids (-100 for non-masked positions)
          'masked_indices'  - positions that were selected as candidates
          'strategy'        - per-candidate strategy: 'mask', 'random', or 'keep'
    """
    if seed is not None:
        np.random.seed(seed)

    token_ids = np.array(token_ids, dtype=np.int64)
    seq_len = len(token_ids)
    input_ids = token_ids.copy()
    # -100 is the standard "ignore in loss" label
    labels = np.full(seq_len, -100, dtype=np.int64)

    # Step 1: Select 15% of positions as masking candidates
    n_candidates = max(1, int(np.round(seq_len * mask_ratio)))
    candidate_indices = np.random.choice(seq_len, size=n_candidates, replace=False)
    candidate_indices.sort()

    # Step 2: Assign strategy to each candidate
    strategies = []
    for idx in candidate_indices:
        r = np.random.random()
        original_token = token_ids[idx]
        labels[idx] = original_token  # store original for loss computation

        if r < 0.80:
            # 80%: replace with [MASK]
            input_ids[idx] = mask_token_id
            strategies.append('mask')
        elif r < 0.90:
            # 10%: replace with a random token (avoid mask_token_id)
            random_tok = np.random.randint(1, vocab_size)
            input_ids[idx] = random_tok
            strategies.append('random')
        else:
            # 10%: keep original unchanged
            input_ids[idx] = original_token
            strategies.append('keep')

    return {
        'input_ids': input_ids.tolist(),
        'labels': labels.tolist(),
        'masked_indices': candidate_indices.tolist(),
        'strategy': strategies
    }


# Demo: apply masking to a 20-token sequence
example_tokens = list(range(1, 21))   # tokens 1..20, vocab_size=100
result = apply_mlm_masking(example_tokens, seed=0)

print("Original tokens :", example_tokens)
print("Masked input_ids:", result['input_ids'])
print()
print(f"Selected {len(result['masked_indices'])} candidates ({len(result['masked_indices'])/len(example_tokens)*100:.0f}%):")
for pos, strat in zip(result['masked_indices'], result['strategy']):
    orig = example_tokens[pos]
    new  = result['input_ids'][pos]
    lbl  = result['labels'][pos]
    print(f"  position {pos:2d}: original={orig:3d}  input_id={new:3d}  label={lbl:3d}  strategy={strat}")

# Verify: strategy distribution should be ~80/10/10
from collections import Counter
counts = Counter(result['strategy'])
print("\nStrategy distribution:", dict(counts))

# Verify round-trip: labels at candidate positions must match original tokens
all_match = all(
    result['labels'][pos] == example_tokens[pos]
    for pos in result['masked_indices']
)
print(f"Label-original match (round-trip check): {all_match}")

## Level 2: Minimal BERT-Like Pretraining and Fine-Tuning in PyTorch

We build a compact transformer (`MiniTransformer`) that mirrors BERT's architecture:
- Token + positional embeddings
- Stacked `TransformerEncoderLayer` blocks
- An MLM prediction head for pretraining

After pretraining on synthetic MLM data we freeze the transformer backbone and attach a
linear classification head on the `[CLS]` token (position 0), then compare downstream
accuracy versus a randomly-initialized baseline.

In [ ]:
# Level 2: MiniTransformer — pretraining + fine-tuning

class MiniTransformer(nn.Module):
    """Minimal BERT-like encoder for MLM pretraining and classification."""
    def __init__(self, vocab_size: int = 100, d_model: int = 64, nhead: int = 4,
                 num_layers: int = 2, max_len: int = 32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=128, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.mlm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [B, T] token ids
        Returns:
            logits: [B, T, vocab_size] for MLM prediction
        """
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        h = self.embed(x) + self.pos_embed(pos)
        h = self.transformer(h)
        return self.mlm_head(h)

    def get_hidden(self, x: torch.Tensor) -> torch.Tensor:
        """Return hidden states [B, T, d_model] without the MLM head."""
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        h = self.embed(x) + self.pos_embed(pos)
        return self.transformer(h)


# ── Synthetic data helpers ──────────────────────────────────────────────────
VOCAB_SIZE_LV2 = 100
SEQ_LEN = 16
MASK_ID = 0

def generate_synthetic_sentences(n: int = 500, seq_len: int = SEQ_LEN,
                                   vocab_size: int = VOCAB_SIZE_LV2) -> np.ndarray:
    """
    Generate token sequences with mild structure:
    position 0 is always the CLS token (id=1), remaining positions are random.
    """
    data = np.random.randint(2, vocab_size, size=(n, seq_len))
    data[:, 0] = 1  # CLS token at position 0
    return data

def batch_mlm_masking(sequences: np.ndarray) -> tuple:
    """
    Apply MLM masking to a batch of sequences.
    Returns (input_ids, labels) as LongTensors.
    """
    inputs, targets = [], []
    for seq in sequences:
        r = apply_mlm_masking(seq.tolist(), vocab_size=VOCAB_SIZE_LV2,
                              mask_token_id=MASK_ID)
        inputs.append(r['input_ids'])
        targets.append(r['labels'])
    return (torch.tensor(inputs, dtype=torch.long),
            torch.tensor(targets, dtype=torch.long))


# ── Pretraining on MLM ──────────────────────────────────────────────────────
pretrain_data = generate_synthetic_sentences(500)
pretrained_model = MiniTransformer(vocab_size=VOCAB_SIZE_LV2).to(device)
optimizer_pt = optim.Adam(pretrained_model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

pretrain_losses = []
pretrained_model.train()
for step in range(100):
    # Sample a mini-batch of 32 sequences each step
    idx = np.random.choice(len(pretrain_data), 32, replace=False)
    batch_seqs = pretrain_data[idx]
    input_ids, labels = batch_mlm_masking(batch_seqs)
    input_ids, labels = input_ids.to(device), labels.to(device)

    logits = pretrained_model(input_ids)          # [B, T, V]
    loss = loss_fn(logits.view(-1, VOCAB_SIZE_LV2), labels.view(-1))

    optimizer_pt.zero_grad()
    loss.backward()
    optimizer_pt.step()
    pretrain_losses.append(loss.item())

print(f"Pretraining done. Initial loss: {pretrain_losses[0]:.4f}  Final loss: {pretrain_losses[-1]:.4f}")


# ── Fine-tuning: pretrained vs random baseline ──────────────────────────────
class ClassifierHead(nn.Module):
    """Linear head over frozen backbone — uses CLS token at position 0."""
    def __init__(self, backbone: MiniTransformer, d_model: int = 64, n_classes: int = 2):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.backbone.get_hidden(x)  # [B, T, D]
        cls = h[:, 0, :]                 # CLS token — first position
        return self.classifier(cls)

def make_cls_dataset(n: int = 200, seq_len: int = SEQ_LEN,
                     vocab_size: int = VOCAB_SIZE_LV2) -> tuple:
    """Binary classification: class determined by whether mean token id > vocab/2."""
    seqs = generate_synthetic_sentences(n, seq_len, vocab_size)
    labels = (seqs[:, 1:].mean(axis=1) > vocab_size / 2).astype(np.int64)
    return torch.tensor(seqs, dtype=torch.long), torch.tensor(labels, dtype=torch.long)


def fine_tune(model_cls, n_epochs: int = 30, lr: float = 1e-3) -> list:
    """Fine-tune classifier head; backbone weights are frozen."""
    for param in model_cls.backbone.parameters():
        param.requires_grad = False
    opt = optim.Adam(model_cls.classifier.parameters(), lr=lr)
    ce = nn.CrossEntropyLoss()
    X_tr, y_tr = make_cls_dataset(200)
    X_tr, y_tr = X_tr.to(device), y_tr.to(device)
    accs = []
    model_cls.train()
    for _ in range(n_epochs):
        logits = model_cls(X_tr)
        loss = ce(logits, y_tr)
        opt.zero_grad(); loss.backward(); opt.step()
        preds = logits.argmax(dim=1)
        accs.append((preds == y_tr).float().mean().item())
    return accs


# Build both classifiers
pretrained_cls = ClassifierHead(pretrained_model).to(device)
random_backbone = MiniTransformer(vocab_size=VOCAB_SIZE_LV2).to(device)  # untrained
random_cls = ClassifierHead(random_backbone).to(device)

pretrained_accs = fine_tune(pretrained_cls)
random_accs = fine_tune(random_cls)

print(f"Pretrained backbone final train acc: {pretrained_accs[-1]:.3f}")
print(f"Random    backbone final train acc:  {random_accs[-1]:.3f}")

## Real-World Example 1: Sentence Embeddings — CLS vs Mean-Pooling

A common downstream task is converting variable-length text into a fixed-size vector.
BERT-style models offer two natural choices:
- **CLS embedding**: use the hidden state at position 0, which was trained to aggregate sequence context
- **Mean-pool embedding**: average all token hidden states — richer coverage but noisier

We extract both, feed them as features to `sklearn.LogisticRegression`, and compare accuracy.

In [ ]:
# Real-World Example 1: CLS vs Mean-Pool sentence embeddings for downstream classification

@torch.no_grad()
def extract_embeddings(model: MiniTransformer,
                       sequences: torch.Tensor,
                       strategy: str = 'cls') -> np.ndarray:
    """
    Extract sentence embeddings from a MiniTransformer.

    Args:
        model: pretrained MiniTransformer
        sequences: [N, T] token id tensor
        strategy: 'cls' (position-0 hidden state) or 'mean' (average all positions)

    Returns:
        embeddings: [N, d_model] numpy array
    """
    model.eval()
    all_embs = []
    batch_size = 32
    for start in range(0, len(sequences), batch_size):
        batch = sequences[start:start+batch_size].to(device)
        h = model.get_hidden(batch)   # [B, T, D]
        if strategy == 'cls':
            emb = h[:, 0, :].cpu().numpy()   # CLS token
        else:
            emb = h.mean(dim=1).cpu().numpy()  # mean over all positions
        all_embs.append(emb)
    return np.vstack(all_embs)


# Build a labelled dataset: 400 examples, binary class
N_EMBED = 400
X_all, y_all = make_cls_dataset(N_EMBED)
split = int(0.8 * N_EMBED)
X_train_t, X_test_t = X_all[:split], X_all[split:]
y_train_np = y_all[:split].numpy()
y_test_np  = y_all[split:].numpy()

# Extract embeddings using pretrained backbone
cls_train  = extract_embeddings(pretrained_model, X_train_t, strategy='cls')
cls_test   = extract_embeddings(pretrained_model, X_test_t,  strategy='cls')
mean_train = extract_embeddings(pretrained_model, X_train_t, strategy='mean')
mean_test  = extract_embeddings(pretrained_model, X_test_t,  strategy='mean')

# Logistic Regression on extracted features (no fine-tuning)
lr_cls  = LogisticRegression(max_iter=500, random_state=42)
lr_mean = LogisticRegression(max_iter=500, random_state=42)
lr_cls.fit(cls_train,  y_train_np)
lr_mean.fit(mean_train, y_train_np)

acc_cls  = accuracy_score(y_test_np, lr_cls.predict(cls_test))
acc_mean = accuracy_score(y_test_np, lr_mean.predict(mean_test))

print(f"Logistic Regression on CLS  embeddings: accuracy = {acc_cls:.3f}")
print(f"Logistic Regression on Mean embeddings: accuracy = {acc_mean:.3f}")
print()
print("Takeaway: CLS is trained to aggregate context (better for classification).")
print("Mean-pool retains all token info (better for semantic similarity tasks).")

# Bar plot comparison
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['CLS', 'Mean-Pool'], [acc_cls, acc_mean], color=['steelblue', 'darkorange'])
ax.set_ylim(0, 1.0)
ax.set_ylabel('Accuracy')
ax.set_title('CLS vs Mean-Pool Embedding Accuracy (Frozen Backbone)')
for i, v in enumerate([acc_cls, acc_mean]):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('/tmp/05_rw1_embeddings.png', dpi=100)
plt.close()
print("Plot saved to /tmp/05_rw1_embeddings.png")

## Real-World Example 2: Fine-Tuning Learning Rate Sensitivity

Choosing the right learning rate when fine-tuning a pretrained model is critical:
- **Too low (1e-5)**: slow convergence, may not adapt within few epochs
- **Too high (1e-2)**: catastrophic forgetting — the pretrained representations are
  overwritten with noise before the task-specific signal can reinforce them

We sweep `lr ∈ {1e-5, 1e-4, 1e-3, 1e-2}`, track train accuracy, and visualise the effect.

In [ ]:
# Real-World Example 2: LR sensitivity sweep — catastrophic forgetting at high LR

def full_fine_tune(backbone: MiniTransformer, lr: float,
                   n_epochs: int = 40, n_samples: int = 200) -> list:
    """
    Fine-tune ALL parameters (backbone + head) — demonstrates forgetting at high LR.

    Args:
        backbone: pretrained or fresh MiniTransformer
        lr: learning rate to use
        n_epochs: number of training epochs
        n_samples: number of training samples

    Returns:
        List of per-epoch training accuracy values
    """
    # Deep-copy weights so we don't mutate the shared backbone
    import copy
    backbone_copy = copy.deepcopy(backbone)
    classifier = nn.Linear(64, 2).to(device)
    all_params = list(backbone_copy.parameters()) + list(classifier.parameters())
    opt = optim.Adam(all_params, lr=lr)
    ce  = nn.CrossEntropyLoss()

    X_tr, y_tr = make_cls_dataset(n_samples)
    X_val, y_val = make_cls_dataset(60)
    X_tr, y_tr   = X_tr.to(device), y_tr.to(device)
    X_val, y_val = X_val.to(device), y_val.to(device)

    val_accs = []
    backbone_copy.train(); classifier.train()
    for epoch in range(n_epochs):
        h = backbone_copy.get_hidden(X_tr)
        logits = classifier(h[:, 0, :])
        loss = ce(logits, y_tr)
        opt.zero_grad(); loss.backward(); opt.step()

        with torch.no_grad():
            backbone_copy.eval(); classifier.eval()
            h_val = backbone_copy.get_hidden(X_val)
            val_logits = classifier(h_val[:, 0, :])
            val_accs.append((val_logits.argmax(1) == y_val).float().mean().item())
            backbone_copy.train(); classifier.train()
    return val_accs


# Build a fresh pretrained backbone for this experiment (avoid mutation above)
import copy
pt_model_for_sweep = copy.deepcopy(pretrained_model)

lr_values = [1e-5, 1e-4, 1e-3, 1e-2]
lr_labels  = ['1e-5', '1e-4', '1e-3', '1e-2']
sweep_results = {}

print("Running LR sweep ...")
for lr, lbl in zip(lr_values, lr_labels):
    accs = full_fine_tune(pt_model_for_sweep, lr=lr, n_epochs=40)
    sweep_results[lbl] = accs
    print(f"  lr={lbl}: best val acc = {max(accs):.3f}  final val acc = {accs[-1]:.3f}")

# Plot learning curves
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['navy', 'steelblue', 'darkorange', 'crimson']
for (lbl, accs), col in zip(sweep_results.items(), colors):
    ax.plot(accs, label=f'lr={lbl}', color=col)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.set_title('Fine-Tuning LR Sensitivity (pretrained backbone)')
ax.legend()
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='random baseline')
plt.tight_layout()
plt.savefig('/tmp/05_rw2_lr_sweep.png', dpi=100)
plt.close()
print("Plot saved to /tmp/05_rw2_lr_sweep.png")
print()
print("High LR (1e-2) causes erratic accuracy — catastrophic forgetting in action.")

## Real-World Example 3: Few-Shot Learning — Pretrained vs Random Init

A major advantage of pretraining is **data efficiency**: the model has already learned
general token patterns, so it requires fewer labelled examples to reach good accuracy.

We fine-tune both a pretrained and a randomly-initialized backbone with
`n_labels ∈ {10, 50, 100, 200}` samples and plot accuracy vs labelled data size.

## Comparison: Random Init vs MLM Pretrained vs Feature Extraction (Frozen)

| Strategy | Training Cost | Labelled Data Needed | Risk of Forgetting |
|---|---|---|---|
| Random init | Full fine-tune | High (needs lots of data) | N/A — no prior knowledge |
| MLM pretrained + full fine-tune | Full fine-tune | Medium | High at large LR |
| Feature extraction (frozen) | Head only | Low | None |

The plot below shows all three strategies across labelled data sizes.

In [ ]:
# Real-World Example 3: Few-shot learning curve + 3-strategy comparison

import copy

def few_shot_accuracy(backbone: MiniTransformer, n_labels: int,
                      freeze: bool = False, n_epochs: int = 50) -> float:
    """
    Fine-tune on n_labels examples; return test accuracy.

    Args:
        backbone: MiniTransformer (pretrained or random)
        n_labels: number of labelled training examples
        freeze: if True, freeze backbone weights (feature extraction)
        n_epochs: epochs to train the head (or full model)

    Returns:
        Test accuracy on 200 held-out examples
    """
    model_copy = copy.deepcopy(backbone).to(device)
    head = nn.Linear(64, 2).to(device)

    if freeze:
        for param in model_copy.parameters():
            param.requires_grad = False
        opt = optim.Adam(head.parameters(), lr=1e-3)
    else:
        opt = optim.Adam(list(model_copy.parameters()) + list(head.parameters()), lr=1e-4)

    ce = nn.CrossEntropyLoss()
    X_tr, y_tr = make_cls_dataset(n_labels)
    X_te, y_te = make_cls_dataset(200)
    X_tr, y_tr = X_tr.to(device), y_tr.to(device)
    X_te, y_te = X_te.to(device), y_te.to(device)

    model_copy.train(); head.train()
    for _ in range(n_epochs):
        h = model_copy.get_hidden(X_tr)
        logits = head(h[:, 0, :])
        loss = ce(logits, y_tr)
        opt.zero_grad(); loss.backward(); opt.step()

    model_copy.eval(); head.eval()
    with torch.no_grad():
        h_te = model_copy.get_hidden(X_te)
        preds = head(h_te[:, 0, :]).argmax(1)
    return (preds == y_te).float().mean().item()


n_label_sizes = [10, 50, 100, 200]
random_init_backbone = MiniTransformer(vocab_size=VOCAB_SIZE_LV2).to(device)

results_pretrained  = []
results_random      = []
results_frozen      = []

print("Few-shot sweep ...")
for n in n_label_sizes:
    acc_pt  = few_shot_accuracy(pretrained_model,   n, freeze=False)
    acc_rnd = few_shot_accuracy(random_init_backbone, n, freeze=False)
    acc_frz = few_shot_accuracy(pretrained_model,   n, freeze=True)
    results_pretrained.append(acc_pt)
    results_random.append(acc_rnd)
    results_frozen.append(acc_frz)
    print(f"  n={n:3d}: pretrained={acc_pt:.3f}  random={acc_rnd:.3f}  frozen={acc_frz:.3f}")

# Plot: accuracy vs number of labelled examples
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_label_sizes, results_pretrained, 'o-', color='steelblue',   label='Pretrained + fine-tune')
ax.plot(n_label_sizes, results_random,     's-', color='crimson',     label='Random init + fine-tune')
ax.plot(n_label_sizes, results_frozen,     '^-', color='darkorange',  label='Frozen (feature extraction)')
ax.set_xlabel('Number of labelled training examples')
ax.set_ylabel('Test Accuracy')
ax.set_title('Few-Shot Learning: Pretrained vs Random Init vs Feature Extraction')
ax.legend()
ax.set_xscale('log')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/05_rw3_fewshot.png', dpi=100)
plt.close()
print("\nPlot saved to /tmp/05_rw3_fewshot.png")
print()
print("Key insight: pretrained model reaches higher accuracy with fewer labelled examples.")
print("Feature extraction (frozen) is competitive at very low data regimes.")
print("Random init needs significantly more data to catch up.")